# Python Bank Account Management System
**Simple case study solution**

This notebook covers account creation, account details, deposit, withdrawal, transfer, pickle file handling, transaction history, NumPy summary statistics, input validation, and an interactive menu.

**Files created when you run it:** `bank_accounts.pkl` (saved account data) and `transaction_history.txt` (transaction history).  
Run the cells from top to bottom. The last cell starts the menu.

In [1]:
# Import required libraries
import pickle
import os
from datetime import datetime
import numpy as np

In [2]:
# File names and starting data
ACCOUNT_FILE = "bank_accounts.pkl"
TRANSACTION_FILE = "transaction_history.txt"

# Load saved accounts if the file already exists
if os.path.exists(ACCOUNT_FILE):
    try:
        with open(ACCOUNT_FILE, "rb") as file:
            accounts = pickle.load(file)
    except (pickle.PickleError, EOFError):
        accounts = {}
else:
    accounts = {}

# Transaction history is stored as one readable line per transaction.
# Existing history is kept; new transactions are appended.
def save_accounts():
    with open(ACCOUNT_FILE, "wb") as file:
        pickle.dump(accounts, file)

def record_transaction(account_number, transaction_type, amount, details=""):
    date_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{date_time} | Account: {account_number} | Type: {transaction_type} | Amount: {amount:.2f} | {details}\n"
    with open(TRANSACTION_FILE, "a", encoding="utf-8") as file:
        file.write(line)

print(f"Loaded {len(accounts)} account(s).")

Loaded 0 account(s).


In [3]:
# Account management functions
def generate_account_number():
    if not accounts:
        return "1001"
    numbers = [int(number) for number in accounts.keys()]
    return str(max(numbers) + 1)

def open_account():
    name = input("Enter account holder name: ").strip()
    if not name:
        print("Name cannot be empty.")
        return

    account_type = input("Enter account type (savings/current): ").strip().lower()
    if account_type not in ["savings", "current"]:
        print("Please enter savings or current.")
        return

    try:
        balance = float(input("Enter initial deposit (0 or more): "))
        if balance < 0:
            print("Initial balance cannot be negative.")
            return
    except ValueError:
        print("Please enter a valid number.")
        return

    account_number = generate_account_number()
    accounts[account_number] = {
        "name": name,
        "type": account_type,
        "balance": balance,
        "transactions": []
    }
    if balance > 0:
        accounts[account_number]["transactions"].append({
            "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "type": "opening deposit",
            "amount": balance
        })
        record_transaction(account_number, "opening deposit", balance)
    save_accounts()
    print(f"Account created successfully. Your account number is {account_number}.")

def view_account():
    account_number = input("Enter account number: ").strip()
    account = accounts.get(account_number)
    if account is None:
        print("Account not found.")
        return
    print("\n--- Account Details ---")
    print("Account holder:", account["name"])
    print("Account number:", account_number)
    print("Account type:", account["type"])
    print(f"Current balance: {account['balance']:.2f}")

In [4]:
# Transaction functions
def get_account(account_number):
    return accounts.get(account_number)

def deposit():
    account_number = input("Enter account number: ").strip()
    account = get_account(account_number)
    if account is None:
        print("Account not found.")
        return
    try:
        amount = float(input("Enter deposit amount: "))
        if amount <= 0:
            print("Amount must be greater than zero.")
            return
    except ValueError:
        print("Please enter a valid amount.")
        return

    account["balance"] += amount
    account["transactions"].append({
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "type": "deposit", "amount": amount
    })
    record_transaction(account_number, "deposit", amount)
    save_accounts()
    print(f"Deposit successful. New balance: {account['balance']:.2f}")

def withdraw():
    account_number = input("Enter account number: ").strip()
    account = get_account(account_number)
    if account is None:
        print("Account not found.")
        return
    try:
        amount = float(input("Enter withdrawal amount: "))
        if amount <= 0:
            print("Amount must be greater than zero.")
            return
    except ValueError:
        print("Please enter a valid amount.")
        return

    if amount > account["balance"]:
        print("Insufficient balance.")
        return

    account["balance"] -= amount
    account["transactions"].append({
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "type": "withdrawal", "amount": amount
    })
    record_transaction(account_number, "withdrawal", amount)
    save_accounts()
    print(f"Withdrawal successful. New balance: {account['balance']:.2f}")

def transfer():
    from_number = input("Enter sender account number: ").strip()
    to_number = input("Enter receiver account number: ").strip()

    if from_number not in accounts or to_number not in accounts:
        print("Sender or receiver account not found.")
        return
    if from_number == to_number:
        print("Sender and receiver must be different accounts.")
        return

    try:
        amount = float(input("Enter transfer amount: "))
        if amount <= 0:
            print("Amount must be greater than zero.")
            return
    except ValueError:
        print("Please enter a valid amount.")
        return

    if amount > accounts[from_number]["balance"]:
        print("Insufficient balance.")
        return

    accounts[from_number]["balance"] -= amount
    accounts[to_number]["balance"] += amount
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    accounts[from_number]["transactions"].append({
        "date": now, "type": "transfer out", "amount": amount
    })
    accounts[to_number]["transactions"].append({
        "date": now, "type": "transfer in", "amount": amount
    })
    record_transaction(from_number, "transfer out", amount, f"To account {to_number}")
    record_transaction(to_number, "transfer in", amount, f"From account {from_number}")
    save_accounts()
    print("Transfer successful.")
    print(f"Sender balance: {accounts[from_number]['balance']:.2f}")
    print(f"Receiver balance: {accounts[to_number]['balance']:.2f}")

In [5]:
# Transaction history and NumPy summary
def transaction_history():
    account_number = input("Enter account number: ").strip()
    account = accounts.get(account_number)
    if account is None:
        print("Account not found.")
        return

    transactions = account["transactions"]
    print(f"\n--- Transaction History: {account_number} ---")
    if not transactions:
        print("No transactions found.")
        return

    for item in transactions:
        print(f"{item['date']} | {item['type']} | Amount: {item['amount']:.2f}")

    amounts = np.array([item["amount"] for item in transactions], dtype=float)
    total_deposits = sum(
        item["amount"] for item in transactions
        if item["type"] in ["deposit", "opening deposit", "transfer in"]
    )
    total_withdrawals = sum(
        item["amount"] for item in transactions
        if item["type"] in ["withdrawal", "transfer out"]
    )

    print("\n--- Summary Statistics ---")
    print(f"Total deposits/incoming transfers: {total_deposits:.2f}")
    print(f"Total withdrawals/outgoing transfers: {total_withdrawals:.2f}")
    print(f"Average transaction amount: {np.mean(amounts):.2f}")
    print(f"Number of transactions: {len(amounts)}")

In [13]:
# Interactive menu
def main_menu():
    while True:
        print("\n========== BANK ACCOUNT MANAGEMENT SYSTEM ==========")
        print("1. Open a new account")
        print("2. View account details")
        print("3. Deposit money")
        print("4. Withdraw money")
        print("5. Transfer money")
        print("6. View transaction history and summary")
        print("7. Exit")

        choice = input("Choose an option (1-7): ").strip()

        if choice == "1":
            open_account()
        elif choice == "2":
            view_account()
        elif choice == "3":
            deposit()
        elif choice == "4":
            withdraw()
        elif choice == "5":
            transfer()
        elif choice == "6":
            transaction_history()
        elif choice == "7":
            save_accounts()
            print("Data saved. Thank you for using the Bank System!")
            break
        else:
            print("Invalid choice. Please select a number from 1 to 7.")

# Run this cell to start the program
main_menu()


========== BANK ACCOUNT MANAGEMENT SYSTEM ==========
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history and summary
7. Exit


Choose an option (1-7):  2
Enter account number:  1004



--- Account Details ---
Account holder: Rahul Goyal
Account number: 1004
Account type: savings
Current balance: 59000.00

========== BANK ACCOUNT MANAGEMENT SYSTEM ==========
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history and summary
7. Exit


Choose an option (1-7):  6
Enter account number:  1004



--- Transaction History: 1004 ---
2026-10-16 15:51:21 | opening deposit | Amount: 100000.00
2026-10-16 15:52:08 | deposit | Amount: 10000.00
2026-10-16 15:52:47 | withdrawal | Amount: 50000.00
2026-10-16 15:53:27 | transfer out | Amount: 1000.00

--- Summary Statistics ---
Total deposits/incoming transfers: 110000.00
Total withdrawals/outgoing transfers: 51000.00
Average transaction amount: 40250.00
Number of transactions: 4

========== BANK ACCOUNT MANAGEMENT SYSTEM ==========
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history and summary
7. Exit


Choose an option (1-7):  78


Invalid choice. Please select a number from 1 to 7.

========== BANK ACCOUNT MANAGEMENT SYSTEM ==========
1. Open a new account
2. View account details
3. Deposit money
4. Withdraw money
5. Transfer money
6. View transaction history and summary
7. Exit


Choose an option (1-7):  7


Data saved. Thank you for using the Bank System!


## Notes
- The program uses `pickle` to load and save account data.
- The transaction text file is opened in append mode (`"a"`), so earlier entries are not overwritten.
- NumPy (`np.mean`) calculates the average transaction amount.
- The optional username/password login feature is not included, to keep the assignment simple.
- Keep the `.ipynb` notebook and generated data files in the same folder so saved accounts can be loaded next time.

In [12]:
print("Current folder:", os.getcwd())

file_path = "bank_accounts.pkl"

if os.path.exists(file_path):
    with open(file_path, "rb") as file:
        accounts = pickle.load(file)

    print("Total accounts:", len(accounts))

    for account_number, details in accounts.items():
        print("\nAccount Number:", account_number)
        print("Name:", details["name"])
        print("Account Type:", details["type"])
        print("Balance:", details["balance"])
else:
    print("Account file not found in this folder.")

Current folder: C:\WS Cube\python\Practise
Total accounts: 4

Account Number: 1001
Name: Chetan Agrawal
Account Type: savings
Balance: 6000.0

Account Number: 1002
Name: Chetan Agrawal
Account Type: savings
Balance: 10000.0

Account Number: 1003
Name: Keshu
Account Type: savings
Balance: 202000.0

Account Number: 1004
Name: Rahul Goyal
Account Type: savings
Balance: 59000.0
